# 01 — Data Ingestion & Cleaning

Loads and cleans the Conti ransomware chat leak (2022) and optionally the Babuk leaks.

**Data source:** The Conti leaks were released publicly in Feb–Mar 2022 by a Ukrainian researcher. Two common structured mirrors:
- GitHub: `https://github.com/TheParmak/conti-leaks-englished` — includes English translations
- Raw JSON archive: search `conti-leaks jabber` on Archive.org or vx-underground

Place raw files in `../data/raw/conti/` before running.

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from langdetect import detect, LangDetectException

RAW_DIR = Path('../data/raw/conti')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load raw data

The leaks exist in two forms: raw JSON (per-conversation files) and flat CSV exports. This cell handles both.

In [ ]:
def load_json_leaks(raw_dir: Path) -> pd.DataFrame:
    """Load Conti-format JSON files — each file is a list of message dicts."""
    records = []
    for f in sorted(raw_dir.glob('**/*.json')):
        with open(f, encoding='utf-8', errors='replace') as fh:
            data = json.load(fh)
        # Handle both list-of-dicts and dict-with-messages-key
        messages = data if isinstance(data, list) else data.get('messages', [])
        for msg in messages:
            msg['_source_file'] = f.name
            records.append(msg)
    return pd.DataFrame(records)


def load_csv_leaks(raw_dir: Path) -> pd.DataFrame:
    dfs = []
    for f in sorted(raw_dir.glob('**/*.csv')):
        dfs.append(pd.read_csv(f, encoding='utf-8', errors='replace'))
    return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()


# Try JSON first, fall back to CSV
json_files = list(RAW_DIR.glob('**/*.json'))
csv_files = list(RAW_DIR.glob('**/*.csv'))

print(f'JSON files found: {len(json_files)}')
print(f'CSV files found:  {len(csv_files)}')

if json_files:
    df_raw = load_json_leaks(RAW_DIR)
elif csv_files:
    df_raw = load_csv_leaks(RAW_DIR)
else:
    raise FileNotFoundError(
        f'No .json or .csv files found in {RAW_DIR}. '
        'Download the Conti leaks and place them there first.'
    )

print(f'\nRaw records loaded: {len(df_raw):,}')
print(df_raw.head(3))

## 2. Normalize schema

Different dumps use different column names. Normalize to a canonical set:
`msg_id, timestamp, sender, recipient, body, source_file`

In [ ]:
# Column name aliases observed across different leak formats
ALIAS = {
    'msg_id':    ['id', 'message_id', 'msg_id', 'ID'],
    'timestamp': ['date', 'ts', 'timestamp', 'time', 'Date'],
    'sender':    ['from', 'fromId', 'from_id', 'sender', 'From', 'author'],
    'recipient': ['to', 'toId', 'to_id', 'recipient', 'To'],
    'body':      ['text', 'body', 'message', 'content', 'msg', 'Text', 'Message'],
}

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    for canonical, aliases in ALIAS.items():
        for alias in aliases:
            if alias in df.columns and canonical not in df.columns:
                rename[alias] = canonical
                break
    df = df.rename(columns=rename)
    # Ensure all canonical columns exist
    for col in ALIAS:
        if col not in df.columns:
            df[col] = None
    if '_source_file' in df.columns:
        df = df.rename(columns={'_source_file': 'source_file'})
    keep = list(ALIAS.keys()) + ['source_file']
    return df[[c for c in keep if c in df.columns]]

df = normalize_columns(df_raw.copy())
print('Columns after normalization:', df.columns.tolist())
print(df.dtypes)

## 3. Parse timestamps & drop null messages

In [ ]:
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)

# Drop records with no message body
before = len(df)
df = df.dropna(subset=['body'])
df = df[df['body'].str.strip() != '']
print(f'Dropped {before - len(df):,} null/empty messages. Remaining: {len(df):,}')

# Deduplicate
before = len(df)
df = df.drop_duplicates(subset=['msg_id'], keep='first') if df['msg_id'].notna().any() else df.drop_duplicates()
print(f'Dropped {before - len(df):,} duplicates. Remaining: {len(df):,}')

df.info()

## 4. Basic text cleaning

In [ ]:
def clean_body(text: str) -> str:
    if not isinstance(text, str):
        return ''
    # Normalize whitespace
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = text.strip()
    return text

df['body_clean'] = df['body'].apply(clean_body)
df['body_len'] = df['body_clean'].str.len()

# Drop messages under 5 chars (noise: ACKs, single chars)
before = len(df)
df = df[df['body_len'] >= 5]
print(f'Dropped {before - len(df):,} very-short messages. Remaining: {len(df):,}')

## 5. Language detection

In [ ]:
def detect_lang(text: str) -> str:
    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'

print('Detecting languages (may take a minute)...')
df['lang'] = df['body_clean'].apply(detect_lang)

lang_counts = df['lang'].value_counts()
print(lang_counts.head(10))

fig, ax = plt.subplots(figsize=(8, 4))
lang_counts.head(8).plot(kind='bar', ax=ax)
ax.set_title('Language distribution — Conti leaks')
ax.set_xlabel('Language code')
ax.set_ylabel('Message count')
plt.tight_layout()
plt.savefig('../data/processed/language_distribution.png', dpi=150)
plt.show()

## 6. Sender activity & timeline

In [ ]:
print('Top 20 senders by message count:')
print(df['sender'].value_counts().head(20))

# Timeline
if df['timestamp'].notna().sum() > 0:
    df_ts = df.set_index('timestamp').sort_index()
    daily = df_ts['body_clean'].resample('D').count()

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(daily.index, daily.values, linewidth=0.8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.xticks(rotation=45)
    ax.set_title('Message volume over time — Conti leaks')
    ax.set_ylabel('Messages per day')
    plt.tight_layout()
    plt.savefig('../data/processed/timeline.png', dpi=150)
    plt.show()
else:
    print('No parseable timestamps found — skipping timeline plot.')

## 7. Save cleaned dataset

In [ ]:
out_path = PROCESSED_DIR / 'conti_cleaned.parquet'
df.to_parquet(out_path, index=False)
print(f'Saved {len(df):,} messages to {out_path}')
print(df[['sender', 'timestamp', 'lang', 'body_len', 'body_clean']].head(5).to_string())